In [ ]:

#imports
import numpy as np
import pandas as pd
import math
import pickle
import os
import ast
import string
import scipy.stats as stats
#no warnings
import warnings
warnings.filterwarnings('ignore')
import json

In [ ]:
#first, get list of participants that are in the study, from xls file 

path_excel = '../data/ParticipantLog.xlsx'

df = pd.read_excel(path_excel, sheet_name='ParticipantLog')

participant_list = []
for ind,row in df.iterrows():
    if row['USE?'] in ['Y','M']:
        participant_list.append(row['participant_id'])
    #print(row['participant_id'], row['USE?'])

print(len(participant_list))
#participant_list

In [ ]:
#get session list, by joining every 2 participants in a single string
session_list = []
for i in range(0,len(participant_list),2):
    session_list.append(str(participant_list[i])+str(participant_list[i+1]))
print(len(session_list))
#session_list

# get vad files per session, merge with openface and opensmile

In [ ]:
#first, selected features from openface
path_openface = '../data/openface/features_extracted/'
path_opensmile  = '../data/opensmile_feats/'
oface_feats = ['timestamp','confidence','success','gaze_0_x','gaze_0_y','gaze_0_z','gaze_1_x','gaze_1_y','gaze_1_z','gaze_angle_x','gaze_angle_y',
               'pose_Tx', 'pose_Ty', 'pose_Tz','pose_Rx', 'pose_Ry', 'pose_Rz', 'AU01_r','AU02_r','AU04_r','AU05_r','AU06_r','AU07_r','AU09_r','AU10_r','AU12_r','AU14_r','AU15_r','AU17_r','AU20_r','AU23_r','AU25_r','AU26_r','AU45_r',
               'AU01_c','AU02_c','AU04_c','AU05_c','AU06_c','AU07_c','AU09_c','AU10_c','AU12_c','AU14_c','AU15_c','AU17_c','AU20_c','AU23_c','AU25_c','AU26_c','AU28_c','AU45_c']
osmile_feats = ['start', 'end', 'F0final_sma', 'voicingFinalUnclipped_sma', 'jitterLocal_sma', 'jitterDDP_sma', 'shimmerLocal_sma', 'logHNR_sma', 'audspec_lengthL1norm_sma', 'audspecRasta_lengthL1norm_sma', 'pcm_RMSenergy_sma', 
                'pcm_zcr_sma', 'audSpec_Rfilt_sma[0]', 'audSpec_Rfilt_sma[1]', 'audSpec_Rfilt_sma[2]', 'audSpec_Rfilt_sma[3]', 'audSpec_Rfilt_sma[4]', 'audSpec_Rfilt_sma[5]', 'audSpec_Rfilt_sma[6]', 'audSpec_Rfilt_sma[7]', 
                'audSpec_Rfilt_sma[8]', 'audSpec_Rfilt_sma[9]', 'audSpec_Rfilt_sma[10]', 'audSpec_Rfilt_sma[11]', 'audSpec_Rfilt_sma[12]', 'audSpec_Rfilt_sma[13]', 'audSpec_Rfilt_sma[14]', 'audSpec_Rfilt_sma[15]', 'audSpec_Rfilt_sma[16]', 
                'audSpec_Rfilt_sma[17]', 'audSpec_Rfilt_sma[18]', 'audSpec_Rfilt_sma[19]', 'audSpec_Rfilt_sma[20]', 'audSpec_Rfilt_sma[21]', 'audSpec_Rfilt_sma[22]', 'audSpec_Rfilt_sma[23]', 'audSpec_Rfilt_sma[24]', 'audSpec_Rfilt_sma[25]', 
                'pcm_fftMag_fband250-650_sma', 'pcm_fftMag_fband1000-4000_sma', 'pcm_fftMag_spectralRollOff25.0_sma', 'pcm_fftMag_spectralRollOff50.0_sma', 'pcm_fftMag_spectralRollOff75.0_sma', 'pcm_fftMag_spectralRollOff90.0_sma', 
                'pcm_fftMag_spectralFlux_sma', 'pcm_fftMag_spectralCentroid_sma', 'pcm_fftMag_spectralEntropy_sma', 'pcm_fftMag_spectralVariance_sma', 'pcm_fftMag_spectralSkewness_sma', 'pcm_fftMag_spectralKurtosis_sma', 
                'pcm_fftMag_spectralSlope_sma', 'pcm_fftMag_psySharpness_sma', 'pcm_fftMag_spectralHarmonicity_sma', 'mfcc_sma[1]', 'mfcc_sma[2]', 'mfcc_sma[3]', 'mfcc_sma[4]', 'mfcc_sma[5]', 'mfcc_sma[6]', 'mfcc_sma[7]', 
                'mfcc_sma[8]', 'mfcc_sma[9]', 'mfcc_sma[10]', 'mfcc_sma[11]', 'mfcc_sma[12]', 'mfcc_sma[13]', 'mfcc_sma[14]']




### checking confidence scores for openface

In [ ]:
#make new folder in featureextraction folder, called merged_feats
path_merged = '../data/merged_feats/'  
path_openface = '../data/openface/features_extracted/'
path_opensmile  = '../data/opensmile_feats/'
if not os.path.exists(path_merged):
    os.mkdir(path_merged)

#now, open vad files for each participant
database = '../data/vad_logs/'
dataconvs = os.listdir(database)

#database for saving participant percentage of frames below 50%
excluded_frames = pd.DataFrame(columns=['session','participant','percentage'])

#session_list = ['1011']

for folder in dataconvs:
    if folder in session_list:

        subf_dir = os.listdir(database + folder)

        log = folder + '_vad.csv'
        path = database + folder + '/' + log
        print(path)
        df = pd.read_csv(path)
        #add session column
        df['session'] = folder


        #now, open openface and opensmile files for each participant
        #openface
        openface_folds = os.listdir(path_openface)
        if len(folder)>4:
            participants = [folder[0:3],folder[3:6]]
        else:
            participants = [folder[0:2],folder[2:4]]
        print(participants) 
        sess_df_openface = pd.DataFrame()

        for participant in participants:
            #find folder
            path_part = path_openface + participant + '/'
            #open csv
            df_openface = pd.read_csv(path_part + participant + '.csv')
            #select only columns of interest
            df_openface = df_openface[oface_feats]
            #now compare timestamps with vad - reduce openface df to only those rows that are in vad,by getting average of columns that are not timestamp
            timestamps_vad = df['timeelapsed'].values

            #print(df_openface.shape, 'BEFORE')
            #df_openface = df_openface[df_openface['confidence']>0.5]
            #print(df_openface.shape, 'AFTER')

            
            new_df_openface = pd.DataFrame()
            for time in timestamps_vad:
                #print(time)
                #get openface rows between time and last time
                if time == timestamps_vad[0]:
                    new_df_openface = pd.concat([new_df_openface,df_openface[df_openface['timestamp']<=time]])
                    last_time = time
                else:
                    temp_openface = df_openface[(df_openface['timestamp']<=time) & (df_openface['timestamp']>last_time)]
                    #print(temp_openface['timestamp'].values[-1])
                    if len(temp_openface)>0:
                        #get average of columns that are not timestamp, except confidence and success, and get max of colunm with "_c" in it
                        for column in temp_openface.columns:
                            if "_c" in column:
                                temp_openface[column] = temp_openface[column].max()
                            elif column not in ['timestamp','confidence','success']:
                                temp_openface[column] = temp_openface[column].mean()
                            if 'timestamp' in column:
                                temp_openface[column] = time
                        #add temp_openface to new_df_openface as a row
                        new_df_openface = pd.concat([new_df_openface.T,temp_openface.iloc[0,:]], axis=1).T
                    else:
                        print('uh oh')
                        print(time, last_time)
                    last_time = time
            print(new_df_openface.shape)

            #now, check the confidence scores for each row -- if they are under 70%, remove them an print the percentage of rows removed
            print(new_df_openface.shape, 'CHECKING CONFIDENCE FOR PARTICIPANT', participant, '...')
            #temp_df_openface = new_df_openface.copy()
            temp_df_openface = new_df_openface[new_df_openface['confidence']>0.7]
            print(new_df_openface.shape, temp_df_openface.shape)
            print('Percentage of rows removed:', (new_df_openface.shape[0]-temp_df_openface.shape[0])/new_df_openface.shape[0])
            excluded_frames = pd.concat([excluded_frames,pd.DataFrame({'session':folder,'participant':participant,'percentage':(new_df_openface.shape[0]-temp_df_openface.shape[0])/new_df_openface.shape[0]},index=[0])])

    excluded_frames

In [ ]:
excluded_frames

### running code

In [ ]:
#make new folder in featureextraction folder, called merged_feats
path_merged = '../data/merged_feats/'  
path_openface = '../data/openface/features_extracted/'
path_opensmile  = '../data/opensmile_feats/'
if not os.path.exists(path_merged):
    os.mkdir(path_merged)

#now, open vad files for each participant
database = '../data/vad_logs/'
dataconvs = os.listdir(database)

#session_list = ['1011']

for folder in dataconvs:
    if folder in session_list:

        subf_dir = os.listdir(database + folder)

        log = folder + '_vad.csv'
        path = database + folder + '/' + log
        print(path)
        df = pd.read_csv(path)
        #add session column
        df['session'] = folder


        #now, open openface and opensmile files for each participant
        #openface
        openface_folds = os.listdir(path_openface)
        if len(folder)>4:
            participants = [folder[0:3],folder[3:6]]
        else:
            participants = [folder[0:2],folder[2:4]]
        print(participants) 
        sess_df_openface = pd.DataFrame()

        for participant in participants:
            #find folder
            path_part = path_openface + participant + '/'
            #open csv
            df_openface = pd.read_csv(path_part + participant + '.csv')
            #select only columns of interest
            df_openface = df_openface[oface_feats]
            #now compare timestamps with vad - reduce openface df to only those rows that are in vad,by getting average of columns that are not timestamp
            timestamps_vad = df['timeelapsed'].values
            
            new_df_openface = pd.DataFrame()
            for time in timestamps_vad:
                #print(time)
                #get openface rows between time and last time
                if time == timestamps_vad[0]:
                    new_df_openface = pd.concat([new_df_openface,df_openface[df_openface['timestamp']<=time]])
                    last_time = time
                else:
                    temp_openface = df_openface[(df_openface['timestamp']<=time) & (df_openface['timestamp']>last_time)]
                    #print(temp_openface['timestamp'].values[-1])
                    if len(temp_openface)>0:
                        #get average of columns that are not timestamp, except confidence and success, and get max of colunm with "_c" in it
                        for column in temp_openface.columns:
                            if "_c" in column:
                                temp_openface[column] = temp_openface[column].max()
                            elif column not in ['timestamp','confidence','success']:
                                temp_openface[column] = temp_openface[column].mean()
                            if 'timestamp' in column:
                                temp_openface[column] = time
                        #add temp_openface to new_df_openface as a row
                        new_df_openface = pd.concat([new_df_openface.T,temp_openface.iloc[0,:]], axis=1).T
                    else:
                        print('uh oh')
                    last_time = time
            print(new_df_openface.shape)

            #now, check the confidence scores for each row -- if they are under 70%, remove them an print the percentage of rows removed
            print(new_df_openface.shape, 'CHECKING CONFIDENCE FOR PARTICIPANT', participant, '...')
            #temp_df_openface = new_df_openface.copy()
            temp_df_openface = new_df_openface[new_df_openface['confidence']>0.7]
            print(new_df_openface.shape, temp_df_openface.shape)
            print('Percentage of rows removed:', (new_df_openface.shape[0]-temp_df_openface.shape[0])/new_df_openface.shape[0])



            #add "_p0" to all columns except timestamp, confidence, success
            for column in new_df_openface.columns:
                if participant == participants[0]:
                    new_df_openface.rename(columns={column:column+'_p0'}, inplace=True)
                else:
                    new_df_openface.rename(columns={column:column+'_p1'}, inplace=True)
            #reindex
            new_df_openface.reset_index(drop=True, inplace=True)
            #add openface df to session df, horizontally, by timestamp (merge horizontally, adding columns, for rows with same timestamp)
            
            sess_df_openface = pd.concat([sess_df_openface,new_df_openface], axis=1)
            print(sess_df_openface.shape)
            
        #now, remove timestamp, confidence, success columns from sess_df_openface
        sess_df_openface.drop(columns=['timestamp_p0','timestamp_p1','confidence_p0','confidence_p1','success_p0','success_p1'], inplace=True)
        print(sess_df_openface.shape)

        #now, add the sess df to the vad df, horizontally, by timestamp (merge horizontally, adding columns, for rows with same timestamp)
        print(df.shape)
        df = pd.concat([df,sess_df_openface], axis=1)
        print(df.shape)

        #now, open opensmile files for each participant
        #opensmile
        sess_df_opensmile = pd.DataFrame()
        opensmile_folds = os.listdir(path_opensmile)
        #print('FILES IN FOLDER', opensmile_folds)

        for file in opensmile_folds:
            for participant in participants:
                if '_'+participant+'.' in file and '#' not in file:
                    #open csv
                    df_opensmile = pd.read_csv(path_opensmile + file)
                    print(df_opensmile.shape)
                    #select only columns of interest
                    df_opensmile = df_opensmile[osmile_feats]
                    #get timestamp column, by changing the "start" column from format 00:00:00.000 (hours, minutes, seconds) to seconds
                    df_opensmile['timestamp'] = df_opensmile['start'].apply(lambda x: sum([a*b for a,b in zip(map(float,x.split(':')), [3600, 60, 1])]))
                    #print(df_opensmile['timestamp'].values)
                    #print(df_opensmile['start'].values)

                    #now compare timestamps with vad - reduce opensmile df to only those rows that are in vad,by getting average of columns that are not timestamp
                    timestamps_vad = df['timeelapsed'].values
                    new_df_opensmile = pd.DataFrame()

                    last_time = 0
                    for time in timestamps_vad:
                        #print(time)
                        #get opensmile rows between time and last time
                        if time == timestamps_vad[0]:
                            new_df_opensmile = pd.concat([new_df_opensmile,df_opensmile[df_opensmile['timestamp']<=time]])
                            last_time = time
                        else:
                            temp_opensmile = df_opensmile[(df_opensmile['timestamp']<=time) & (df_opensmile['timestamp']>last_time)]
                            #print(temp_opensmile['timestamp'].values[-1])
                            if len(temp_opensmile)>0:
                                #get average of columns that are not timestamp, except confidence and success, and get max of colunm with "_c" in it
                                for column in temp_opensmile.columns:
                                    if column not in ['timestamp','start','end']:
                                        temp_opensmile[column] = temp_opensmile[column].mean()
                                    if 'timestamp' in column:
                                        temp_opensmile[column] = time
                                #add temp_opensmile to new_df_opensmile as a row
                                new_df_opensmile = pd.concat([new_df_opensmile.T,temp_opensmile.iloc[0,:]], axis=1).T
                            else:
                                print('uh oh')
                            last_time = time
                    print(new_df_opensmile.shape)
                    #add "_p0" to all columns except timestamp, start,end
                    for column in new_df_opensmile.columns:
                        if participant == participants[0]:
                            new_df_opensmile.rename(columns={column:column+'_p0'}, inplace=True)
                        else:
                            new_df_opensmile.rename(columns={column:column+'_p1'}, inplace=True)
                    #reindex    
                    new_df_opensmile.reset_index(drop=True, inplace=True)
                    #add opensmile df to session df, horizontally, by timestamp (merge horizontally, adding columns, for rows with same timestamp)
                    sess_df_opensmile = pd.concat([sess_df_opensmile,new_df_opensmile], axis=1)
                    print(sess_df_opensmile.shape)
        #now, remove timestamp, start, end columns from sess_df_opensmile
        sess_df_opensmile.drop(columns=['timestamp_p0','timestamp_p1','start_p0','start_p1','end_p0','end_p1'], inplace=True)
        print(sess_df_opensmile.shape)

        #now, add the sess df to the vad df, horizontally, by timestamp (merge horizontally, adding columns, for rows with same timestamp)
        print(df.shape)
        df = pd.concat([df,sess_df_opensmile], axis=1)
        print(df.shape)

        #now, save the merged df as a csv
        df.to_csv(path_merged + folder + '_merged.csv', index=False)
        



In [ ]:
df

# get jsons again, correct interpolation

In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.interpolate import interp1d
#import mse
from sklearn.metrics import mean_squared_error
from scipy.signal import medfilt


In [ ]:
#one by one, open json files in a folder, read them and add them to pandas dataframe as two columns: 'time' and 'value'

#see files in folder
filelist = os.listdir('../data/ParticipantsJSON/')
filelist = [file for file in filelist if file.endswith('.json')]

#make empty dataframe
df = pd.DataFrame(columns=['participant_number','session','timestamp', 'frame_number','frame_seconds','seconds', 'value'], index=(range(len(participant_list))))
dfrow = 0
#open files one by one and read
for file in filelist:
    pnumber = int(file[:-12])
    print(pnumber)
    if pnumber in participant_list:
        
        #session number, knowing it started at 0 and it's 2 participants per session (p number start at 0)
        session = np.ceil((pnumber-1)/2)

        # Opening JSON file
        f = open('../data/ParticipantsJSON/' + file)
        
        # returns JSON object as 
        # a dictionary
        data = json.load(f)
        #print(data)
    
        #adding to pandas df, one column for time==keys and one for values, but a single row
        #get dictionary keys
        keys = list(data.keys())
        #get dictionary values
        values_list = list(data.values())
        print(len(keys))
        values = []
        #turn every value into an int
        for value in values_list:
            value = int(value)
            values.append(value)
            
        if keys[0] == "ID":
            keys = keys[1:]
            values = values[1:]
        frames = []
        seconds = []
        frameseconds = []
        for keyf in keys:
            #split keyf according to ":"
            keyf = keyf.split(":")
            #get frames (last)
            frames.append(int(keyf[-1]))
            frameseconds.append(int(keyf[-1])/25)
            #get seconds (second to last all the way to the first)
            seconds.append(int(keyf[-2]) + 60 * int(keyf[-3]) + 3600 * int(keyf[-4]))

            
        #place in df
        df.loc[dfrow] = [pnumber,session, keys, frames,frameseconds,seconds,values]
        dfrow += 1

        # Closing file
        f.close()

df = df.sort_values(by=['participant_number'])
df.index = range(len(df))
#save df as csv
df.to_csv('../data/participant_json.csv', index=False)
df

In [ ]:
#for each session, calculate degree of disagreement between participants with a scale of 0.1 seconds

#create new df with columns: session, participant, frame_seconds, value
df_interp = pd.DataFrame(columns=['session','participant_number','frame_seconds','value'], index=range(len(df)))

#first, get session df
sessions = df['session'].unique()
for session in sessions:
    df_session = df[df['session']==session]
    #get all participants
    participants = df_session['participant_number'].unique()
    #per participant, get all frame_seconds
    for participant in participants:
        df_p = df_session[df_session['participant_number']==participant]
  
        frame_seconds = df_p['frame_seconds'].values[0]
        values = df_p['value'].values[0]
        #now, interpolate both arrays to have the frames_seconds as the same at a 0.1 second interval
        #get min and max frame_seconds
        min_frame_seconds = min(frame_seconds)
        max_frame_seconds = np.round(max(frame_seconds),1) 
    
        
        #create new frame_seconds array
        new_frame_seconds = np.arange(0.0,max_frame_seconds,0.5) #changed from 0.1 to 0.5

        print(participant,min_frame_seconds,max_frame_seconds,len(new_frame_seconds),len(frame_seconds))
    
        #interpolate values (step interpolation)
        #Create the interpolation function
        f = interp1d(frame_seconds, values, kind='next',fill_value='extrapolate')
        # Use the interpolation function to get the corresponding y-values
        new_values = f(new_frame_seconds)

        #print('new_session')
        #print(len(new_frame_seconds),len(new_values))
        #print(len(frame_seconds),len(values))
        
        #add to new df
        df_interp.loc[df_p.index] = [session,participant,new_frame_seconds,new_values]

    #print(df_interp.head())

    #just to check interpolation:
    ##plot for one partipant, then plot the other in the session
    #plt.figure(figsize=(10,5))
    ##plot from values in df_interp
    #for p in [participant -1, participant]:
    #    df_p = df_interp[df_interp['participant_number']==p]
    #    x = np.array(df_p['frame_seconds'].values[0])
    #    y = np.array(df_p['value'].values[0])
    #    plt.plot(x,y,label = 'Participant ' + str(p))
    #    #also plot the original values
    #    df_p = df_session[df_session['participant_number']==p]
    #    x = np.array(df_p['frame_seconds'].values[0])
    #    y = np.array(df_p['value'].values[0])
    #    plt.plot(x,y,label = 'Participant ' + str(p) + ' original')

    #plt.title('Session ' + str(session))
    #plt.legend()
    #plt.xlabel('Frame seconds')
    #plt.ylabel('Value')
    #plt.show()

deltas = []
delta_signs = []
cumsums = []
deltacumsum = []
    
#go back to df_interp, session by session, and make the frames_seconds be the same for all participants (equal to the max length of the frames_seconds array)
#and then extrapolate the values as per the last value
sessions = df_interp['session'].unique()
for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    
    #get max length of frames_seconds
    max_length = max([len(x) for x in df_session['frame_seconds'].values])
    max_time = max([max(x) for x in df_session['frame_seconds'].values])
    min_time = min([min(x) for x in df_session['frame_seconds'].values])
   
    #get all participants
    participants = df_session['participant_number'].unique()
    #per participant, get all frame_seconds
    for participant in participants:
        old_frame = df_session[df_session['participant_number']==participant]['frame_seconds'].values[0]
        old_values = df_session[df_session['participant_number']==participant]['value'].values[0]
        #create new frame_seconds array
        new_frame_seconds = np.arange(0.0,max_time+0.5,0.5)[:max_length] #CHANGERD FROM 0.1 to 0.5
        #print('participant 1 ', max_time, np.array_equal(new_frame_seconds,old_frame))
        #if needed, extend old_values to have the same length as new_frame_seconds, by the last value
        if len(old_frame) < max_length:
            old_values = np.append(old_values,[old_values[-1]]*(max_length-len(old_frame)))
        #replace in df_interp
        df_interp.loc[df_session[df_session['participant_number']==participant].index] = [session,participant,new_frame_seconds,old_values]
        print('interp')
        print(old_frame.shape,new_frame_seconds.shape,old_values.shape)

    #add information about the derivative of the values
    #get all participants

    df_session = df_interp[df_interp['session']==session]
    participants = df_session['participant_number'].unique()
    
    #per participant, get all frame_seconds
    for participant in participants:
        print(participant)
        old_frame = df_session[df_session['participant_number']==participant]['frame_seconds'].values[0]
        old_values = df_session[df_session['participant_number']==participant]['value'].values[0]
        #calculate delta between every to consequentent values
        delta_values = np.diff(old_values)
        deltas.append(delta_values)
        #calculate how derivative changes over time - if positive,1, if negative, -1, if 0, 0
        delta_sign = np.sign(delta_values)
        delta_signs.append(delta_sign)

        #calculate cumsum
        cumsum_values = np.cumsum(old_values)
        cumsums.append(cumsum_values)
        #calculate values with median filter of size 101 (10 seconds)
        median_values = medfilt(old_values,101)
        deltacumsum.append(median_values)

    


#add to df_interp
df_interp['cumsum'] = cumsums
df_interp['derivative'] = deltas
df_interp['derivative_sign'] = delta_signs
df_interp['medianfilt'] = deltacumsum

#print the range of values in the deltas
delta_signs = np.concatenate(delta_signs)
print('Delta signs:', np.unique(delta_signs))
#print counts
unique, counts = np.unique(delta_signs, return_counts=True)
print('Counts:', dict(zip(unique, counts)))



#save df_interp
df_interp.to_pickle('../df_interpolated5dec.pkl')
#to csv
df_interp.to_csv('../df_interpolated5dec.csv')



# open log dfs and process total turns, floorhold, etc

In [ ]:
#one by one open merged files
path_merged = '../data/merged_feats/'
merged_files = os.listdir(path_merged)
merged_files = [file for file in merged_files if file.endswith('.csv')]

new_folder = '../data/merged_feats_vad/'
if not os.path.exists(new_folder):
    os.mkdir(new_folder)

for file in merged_files:
    df = pd.read_csv(path_merged + file)

    total_talking_time_p0 = []
    total_number_turns_p0 = []
    total_talking_time_p1 = []
    total_number_turns_p1 = []
    balance_interaction = []
    
    #replace False with "None" in "talking_now" and "talking_before" columns
    df[['talking_now', 'talking_before']] = df[['talking_now', 'talking_before']].replace(False, "None")

    #right is participant with the < number
    #left is participant with the > number
    #get total talking time for each participant, as rows progress
    for ind,row in df.iterrows():
      if ind == 0:
        old_talk_p0 = 0
        old_talk_p1 = 0
      if row['talking_now'] == 'right':
          total_talking_time_p0.append(old_talk_p0 + row['timeintervals'])
          total_talking_time_p1.append(old_talk_p1)
          old_talk_p0 = old_talk_p0 + row['timeintervals']
      elif row['talking_now'] == 'left':
          total_talking_time_p1.append(old_talk_p1 + row['timeintervals'])
          total_talking_time_p0.append(old_talk_p0)
          old_talk_p1 = old_talk_p1 + row['timeintervals']
      else:
          total_talking_time_p0.append(old_talk_p0)
          total_talking_time_p1.append(old_talk_p1)
      total_talking_time_both = old_talk_p0 + old_talk_p1
      if total_talking_time_both == 0:
        floorhold_p0 = 0
        floorhold_p1 = 0
        balance_p0 = 0
        balance_p1 = 0

      else:
        floorhold_p0 = old_talk_p0/total_talking_time_both
        floorhold_p1 = old_talk_p1/total_talking_time_both
        #get balance
        balance_p0 = (floorhold_p0/(floorhold_p0 + floorhold_p1))
        balance_p1 = (floorhold_p1/(floorhold_p0 + floorhold_p1))


      #get imbalance
      imbalance = np.abs(balance_p0 - 0.5) + np.abs(balance_p1 - 0.5)  #from https://dl.acm.org/doi/10.1145/3434074.3447070
      balance_interaction.append(imbalance)
    

          
      
    #print(total_talking_time_p0)
    #cluster df by sequential rows with the same value for "talking_now" (using groupby)

    df_cluster = df.groupby((df['talking_now'] != df['talking_now'].shift()).cumsum()).agg({'globaltime': ['first', 'last'], 'timeelapsed': ['first', 'last'], 'talking_now': ['first'], 'talking_before': ['first']})

    #now, row by row, get the number of turns for each participant, based on each cluster
    #save the speaker, and the globaltime of the first and last row of the cluster
    speaker = []
    globaltime_first = []
    globaltime_last = []
    tot_left = 0
    tot_right = 0
    total_turns_speaker = []
    for ind,row in df_cluster.iterrows():
      speaker.append(row['talking_now']['first'])
      #print(row['talking_now']['first'])
      #add total turns per speaker, based on who's the speaker
      if row['talking_now']['first'] == 'right':
        tot_right += 1
        total_turns_speaker.append(tot_right)
      elif row['talking_now']['first'] == 'left':
        tot_left += 1
        total_turns_speaker.append(tot_left)
      else:
        total_turns_speaker.append(0)
      globaltime_first.append(row['globaltime']['first'])
      globaltime_last.append(row['globaltime']['last'])

      #print(row['globaltime']['first'],row['globaltime']['last'])
    #print(df['globaltime'].values[0],df['globaltime'].values[-1])

    totalturns_p0 = 0
    totalturns_p1 = 0
    #now, go row by row to the merged df, add number of turns as many times as the cluster lasts
    for ind,row in df.iterrows():
      time = row['globaltime']
      #print(time)
      #find the index of the cluster that contains this time
      for i in range(len(globaltime_first)):
        if time >= globaltime_first[i] and time <= globaltime_last[i]:
          
          #add number of turns to df
          if speaker[i] == 'right':
            totalturns_p0 = total_turns_speaker[i]

            total_number_turns_p0.append(totalturns_p0)
            total_number_turns_p1.append(totalturns_p1)
       
          elif speaker[i] == 'left':
            totalturns_p1 = total_turns_speaker[i]
            total_number_turns_p0.append(totalturns_p0)
            total_number_turns_p1.append(totalturns_p1)
          else:
            total_number_turns_p0.append(totalturns_p0)
            total_number_turns_p1.append(totalturns_p1)
        
    #move session column to the beginning
    cols = list(df.columns)
    cols = [cols[8]] + cols[0:8] + cols[9:]
    df = df[cols]
    #print(df.columns)
    #move timeelapsed column to place 2
    cols = list(df.columns)
    cols = [cols[0]] + [cols[8]] + cols[1:8] + cols[9:]
    df = df[cols]
    #print(df.columns)

    #remove some columns
    df = df.drop(columns=['left_speech_active','right_speech_active','talking_before','logtype'])
    

    #add to df in specific column order

    df['total_talking_time_p0'] = total_talking_time_p0
    df['total_talking_time_p1'] = total_talking_time_p1
    df['total_number_turns_p0'] = total_number_turns_p0
    df['total_number_turns_p1'] = total_number_turns_p1
    df['imbalance'] = balance_interaction

    #reorder columns
    cols = list(df.columns)
    cols = cols[0:5] + cols[-5:] + cols[5:-5]
    df = df[cols]
    #print(df.columns)

    df.to_csv(new_folder + file[:-4]+'_vad.csv', index=False)
    
    
    df
    



In [ ]:
#right is participant with the < number
#left is participant with the > number

# open jsons and get right precision

In [ ]:
#open json csv
path_json = '../df_interpolated5dec.pkl'
df_json = pd.read_pickle(path_json)
df_json


In [ ]:
#session by session of the merged dfs, get the json data exploded into a new dataframe

new_path_merged = '../data/merged_feats_gt5'
#make dir
if not os.path.exists(new_path_merged):
    os.mkdir(new_path_merged)

path_merged = '../data/merged_feats_vad/'
merged_folds = os.listdir(path_merged)
merged_folds = [file for file in merged_folds if file.endswith('.csv')]
for folder in merged_folds:
    session = folder[:-15]
    print(session)
    if len(session)>4:
        participants = [session[0:3],session[3:6]]
    else:
        participants = [session[0:2],session[2:4]]
    print(participants)
    #open merged csv
    df_merged = pd.read_csv(path_merged + folder)
    #find row corresponding to each participant
    long_json_both = pd.DataFrame()
    for participant in participants:
        long_json = pd.DataFrame()
        df_jsonp = df_json[df_json['participant_number']==int(participant)]
        #convert string of array to array
        #df_jsonp['frame_seconds'] = df_jsonp['frame_seconds'].apply(lambda x: ast.literal_eval(x))
        #df_jsonp['values'] = df_jsonp['values'].apply(lambda x: ast.literal_eval(x))
        #df_jsonp['cumsum'] = df_jsonp['cumsum'].apply(lambda x: ast.literal_eval(x))
        #df_jsonp['derivative'] = df_jsonp['derivative'].apply(lambda x: ast.literal_eval(x))
        #print(df_jsonp)
        #now, explode the rows of the json data - frame_seconds, values, cumsum, derivative
        long_json['timestamp'] = df_jsonp['frame_seconds'].explode().values
        long_json['value'] = df_jsonp['value'].explode().values
        long_json['cumsum'] = df_jsonp['cumsum'].explode().values
        derivatives = df_jsonp['derivative'].explode().values
        #add zero at start
        derivatives = np.insert(derivatives,0,0.0)
        long_json['derivative'] = derivatives
        #now for derivative signs
        derivative_signs = df_jsonp['derivative_sign'].explode().values
        derivative_signs = np.insert(derivative_signs,0,0)
        long_json['derivative_sign'] = derivative_signs
        #print(long_json.shape)
        #add "_p0" to all columns except timestamp
        for column in long_json.columns:
            if participant == participants[0]:
                long_json.rename(columns={column:column+'_p0'}, inplace=True)
            else:
                long_json.rename(columns={column:column+'_p1'}, inplace=True)
        #reindex
        long_json.reset_index(drop=True, inplace=True)
        #add long_json to long_json_both, horizontally, by timestamp (merge horizontally, adding columns, for rows with same timestamp)
        long_json_both = pd.concat([long_json_both,long_json], axis=1)
    #print(long_json_both.head())
    #print latest time
    #print(df_merged['timeelapsed'].values[-1])
    #print(long_json_both['timestamp_p0'].values[-1])
    #now, add the long_json_both to the merged df, horizontally, by timestamp (merge horizontally, adding columns, for rows with same timestamp)
    #you should select first only the rows of the timestamps in the long_json_both that are <= the last timestamp in the merged df
    timestamps_merged = df_merged['timeelapsed'].values
    last_time = 0
    new_undersampled_df = pd.DataFrame(columns=long_json_both.columns)

    for time in timestamps_merged:
        #print(time)
        if time == timestamps_merged[0]:
            small_json = long_json_both[long_json_both['timestamp_p0']<=time]
        else:
            small_json = long_json_both[(long_json_both['timestamp_p0']<=time) & (long_json_both['timestamp_p0']>last_time)]
        last_time = time
        if small_json.shape[0]<1:
            
            print('uh oh')
        else:
            #replace the time with the time in the merged df
            small_json['timestamp_p0'] = time
            small_json['timestamp_p1'] = time
            #add last value of small_json to new_undersampled_df
            new_undersampled_df = pd.concat([new_undersampled_df,small_json.iloc[-1:,:]])
    print(new_undersampled_df.shape)
    #now, remove timestamp columns from new_undersampled_df
    new_undersampled_df.drop(columns=['timestamp_p0','timestamp_p1'], inplace=True)
    print(new_undersampled_df.shape)
    #reindex
    new_undersampled_df.reset_index(drop=True, inplace=True)
    #now, add the new_undersampled_df to the merged df, horizontally, by timestamp (merge horizontally, adding columns, for rows with same timestamp)
    df_merged = pd.concat([df_merged,new_undersampled_df], axis=1)
    print(df_merged.shape) 
    #add column that is difference between value_p0 and value_p1
    df_merged['value_diff'] = np.abs(df_merged['value_p0'] - df_merged['value_p1'])
    #now, sum values
    df_merged['value_sum'] = df_merged['value_p0'] + df_merged['value_p1']
    #add column that is difference between cumsum_p0 and cumsum_p1
    df_merged['cumsum_diff'] = np.abs(df_merged['cumsum_p0'] - df_merged['cumsum_p1'])
    #now, sum cumsums
    df_merged['cumsum_sum'] = df_merged['cumsum_p0'] + df_merged['cumsum_p1']
    #for each value of cumsum_p0 and cumsum_p1, adjust a linear regression and get slope
    slope_p1 = []
    slope_p0 = []
    #fill nan with 0 for cumsum
    df_merged['cumsum_p0'] = df_merged['cumsum_p0'].fillna(0)
    df_merged['cumsum_p1'] = df_merged['cumsum_p1'].fillna(0)
    for ind,row in df_merged.iterrows():
        if ind < 2:
            slope_p1.append(0)
            slope_p0.append(0)
        else:
            #print(row['cumsum_p0'],row['cumsum_p1'])
            datap0 = np.array(df_merged['cumsum_p0'].values[:ind])
            x = np.array(df_merged['timeelapsed'].values[:ind])
            #print(x.shape, datap0.shape)
            slope, intercept, r_value, p_value, std_err = stats.linregress(x,datap0)
            slope_p0.append(slope)
            datap1 = np.array(df_merged['cumsum_p1'].values[:ind])
            slope, intercept, r_value, p_value, std_err = stats.linregress(x,datap1)
            slope_p1.append(slope)
        #if ind == df_merged.shape[0]-1:
        #    slope_p1.append(slope_p1[-1])
        #    slope_p0.append(slope_p0[-1])
    df_merged['slope_p0'] = slope_p0
    df_merged['slope_p1'] = slope_p1
    #difference between slopes
    df_merged['slope_diff'] = np.abs(df_merged['slope_p0'] - df_merged['slope_p1'])


    #now, for colum "talking_now", create 2 columns: one that is 1 when right is talking, and one that is 1 when left is talking
    df_merged['vad_p0'] = df_merged['talking_now'].apply(lambda x: 1 if x == 'right' else 0)
    df_merged['vad_p1'] = df_merged['talking_now'].apply(lambda x: 1 if x == 'left' else 0)
    #move these columns to close to talking_now (position 5)
    cols = list(df_merged.columns)
    cols = cols[0:5] + cols[-2:] + cols[5:-2]
    #drop talking ow
    df_merged.drop(columns=['talking_now'], inplace=True)

    #now, change the columns for groundtruth so they are in places 3-onwards
    cols = list(df_merged.columns)
    cols = cols[0:4] + cols[-15:] + cols[4:-15]
    df_merged = df_merged[cols]
    print(cols)

    #now, save the merged df as a csv
    df_merged.to_csv(new_path_merged + '/' + session+ '_merged_gt.csv', index=False)
    





# merge all and rename

In [ ]:
#one by one, open merged files
path_merged = '../data/merged_feats_gt5/'
merged_files = os.listdir(path_merged)
merged_files = [file for file in merged_files if file.endswith('.csv')]
#if there is no training folder, make one
if not os.path.exists('../data/training/'):
    os.mkdir('../data/training/')

all_data = pd.DataFrame()

for file in merged_files:
    df = pd.read_csv(path_merged + file)
    df = df.drop(columns=['globaltime','timeintervals', 'cumsum_p0', 'derivative_p0', 
                          'cumsum_p1', 'derivative_p1', 'value_diff', 'value_sum', 'cumsum_diff', 
                          'cumsum_sum', 'slope_p0', 'slope_p1']
                    )
    
    #now, all columns with p0 in the name should come after columns 2, and then be followed by all with p1
    cols = list(df.columns)
    cols_p0 = [col for col in cols if ('_p0' in col) and 'value_p0' not in col]
    cols_p1 = [col for col in cols if ('_p1' in col) and 'value_p1' not in col]
    df['groundtruth'] = df['value_p0']
    #sessio is now session_0
    df['session'] = df['session'].apply(lambda x: str(x) + '_0')
    cols = cols[0:2] + ['groundtruth','value_p0','value_p1','imbalance'] + cols_p0 + cols_p1
    print(cols)
    df = df[cols]
    #now, copy this df, change columns to have p1 first, and then paste it after the original df
    df_p1 = df.copy()
    #session_number is now session_1
    df_p1['session'] = df_p1['session'].apply(lambda x: str(x)[:-1] + '1')
    cols = list(df_p1.columns)
    cols_p0 = [col for col in cols if '_p0' in col and 'value_p0' not in col]
    cols_p1 = [col for col in cols if '_p1' in col and 'value_p1' not in col]
    df_p1['groundtruth'] = df['value_p1']
    cols = cols[0:2] + ['groundtruth','value_p0','value_p1','imbalance'] + cols_p1 + cols_p0
    df_p1 = df_p1[cols]
    print(df.shape)
    #now, add df_p1 to df
    df = pd.concat([df,df_p1], axis=0)
    print(df.shape)

    #now, rename columns - columns from col 4 onwards should be "s" and then a number
    cols = list(df.columns)
    #print(cols)

    #new_cols = cols[0:5] + ['s' + str(i) for i in range(1,len(cols)-4)]
    #df.columns = new_cols
    #print(df.columns)
    #create dictionary with old and new column names
    dict_cols = {}
    for i in range(len(cols)):
        dict_cols[cols[i]] = new_cols[i]
    #save
    with open('../data/training/dict_cols.json', 'w') as fp:
        json.dump(dict_cols, fp)



    #now, concatenate in the full database
    all_data = pd.concat([all_data,df], axis=0)

#remove all the NaN and inf
print(all_data.shape)
all_data = all_data.replace([np.inf, -np.inf], np.nan)
all_data = all_data.dropna()
print(all_data.shape)

#save all_data
all_data.to_csv('../data/training/all_data_05.csv', index=False)

all_data


In [ ]:
#check for nan and inf
print(all_data.shape)
print(all_data.isnull().sum())
print(all_data.isin([np.inf, -np.inf]).sum())
#remove all the NaN and inf
all_data = all_data.replace([np.inf, -np.inf], np.nan)
all_data = all_data.dropna()
print(all_data.shape)
#save all_data
all_data.to_csv('../data/training/all_data_05.csv', index=False)

In [ ]:
#now, normalize it through standard scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

#fit and transform
all_data.iloc[:,5:] = scaler.fit_transform(all_data.iloc[:,5:])

#replace nan and inf
print(all_data.shape)
all_data = all_data.replace([np.inf, -np.inf], np.nan)
all_data = all_data.dropna()
print(all_data.shape)

#save scaler
import joblib
joblib.dump(scaler, '../data/training/scaler.pkl')
all_data.to_csv('../data/training/all_data_05_norm.csv', index=False)
all_data

In [ ]:
dict_cols

In [ ]:
# merge all and rename ................FOR DERIVATIVE SIGNS
#one by one, open merged files
path_merged = '../data/merged_feats_gt5/'
merged_files = os.listdir(path_merged)
merged_files = [file for file in merged_files if file.endswith('.csv')]
#if there is no training folder, make one
if not os.path.exists('../data/training/'):
    os.mkdir('../data/training/')

all_data = pd.DataFrame()

for file in merged_files:
    df = pd.read_csv(path_merged + file)
    df = df.drop(columns=['globaltime','timeintervals', 'cumsum_p0', 'derivative_p0', 
                          'cumsum_p1', 'derivative_p1', 'value_diff', 'value_sum', 'cumsum_diff', 
                          'cumsum_sum', 'slope_p0', 'slope_p1']
                    )
    
    #now, all columns with p0 in the name should come after columns 2, and then be followed by all with p1
    cols = list(df.columns)
    cols_p0 = [col for col in cols if ('_p0' in col) and 'derivative_sign_p0' not in col]
    cols_p1 = [col for col in cols if ('_p1' in col) and 'derivative_sign_p1' not in col]
    #change session to session_0
    df['session'] = df['session'].apply(lambda x: str(x) + '_0')
    df['groundtruth'] = df['derivative_sign_p0']
    cols = cols[0:2] + ['groundtruth','derivative_sign_p0','derivative_sign_p1','imbalance'] + cols_p0 + cols_p1
    print(cols)
    df = df[cols]
    #now, copy this df, change columns to have p1 first, and then paste it after the original df
    df_p1 = df.copy()
    #change session to session_1
    df_p1['session'] = df_p1['session'].apply(lambda x: str(x)[:-1] + '1')
    cols = list(df_p1.columns)
    cols_p0 = [col for col in cols if '_p0' in col and 'derivative_sign_p0' not in col]
    cols_p1 = [col for col in cols if '_p1' in col and 'derivative_sign_p1' not in col]
    df_p1['groundtruth'] = df['derivative_sign_p1']
    cols = cols[0:2] + ['groundtruth','derivative_sign_p0','derivative_sign_p1','imbalance'] + cols_p1 + cols_p0
    df_p1 = df_p1[cols]
    print(df.shape)
    #now, add df_p1 to df
    df = pd.concat([df,df_p1], axis=0)
    print(df.shape)

    #now, rename columns - columns from col 4 onwards should be "s" and then a number
    cols = list(df.columns)
    print(cols)

    new_cols = cols[0:5] + ['s' + str(i) for i in range(1,len(cols)-4)]
    df.columns = new_cols
    print(df.columns)
    #create dictionary with old and new column names
    dict_cols = {}
    for i in range(len(cols)):
        dict_cols[cols[i]] = new_cols[i]
    #save
    with open('../data/training/dict_cols_sign.json', 'w') as fp:
        json.dump(dict_cols, fp)



    #now, concatenate in the full database
    all_data = pd.concat([all_data,df], axis=0)

#remove groundtruth = 0 rows

all_data = all_data[all_data['groundtruth']!=0]
#make -1 values be 0 for the groundtruth column
all_data['groundtruth'] = all_data['groundtruth'].replace(-1,0)

print(df.columns)

#remove all the NaN and inf
print(all_data.shape)
all_data = all_data.replace([np.inf, -np.inf], np.nan)
all_data = all_data.dropna()
print(all_data.shape)

#save all_data
all_data.to_csv('../data/training/all_data_05_sign.csv', index=False)



#now, normalize it through standard scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()



#fit and transform
all_data.iloc[:,5:] = scaler.fit_transform(all_data.iloc[:,5:])

#remove nan
print(all_data.shape)
all_data = all_data.replace([np.inf, -np.inf], np.nan)
all_data = all_data.dropna()
print(all_data.shape)

#save scaler
import joblib
joblib.dump(scaler, '../data/training/scaler_sign.pkl')
all_data.to_csv('../data/training/all_data_05_norm_sign.csv', index=False)
all_data


In [ ]:
#now, one by one, open the dfs and then check label imbalance. plot
dfs = ['all_data_05.csv','all_data_05_norm.csv']

# plots

In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.interpolate import interp1d
#import mse
from sklearn.metrics import mean_squared_error
from scipy.signal import medfilt


In [ ]:
#open df_turns
df_turns = pd.read_csv('../all_merged_df.csv')
#open df_turns_diff
df_turns_diff = pd.read_csv('../df_turns_diff.csv')

In [ ]:
#print participant demographics
#avg age + std
df_turns['age'] = df_turns['age'].astype(float)
print('Average age: ',df_turns['age'].mean(), ' +- ', df_turns['age'].std())
#age range
print('Age range: ',df_turns['age'].min(), ' - ', df_turns['age'].max())
#gender distribution - count occurrences of every value
print( df_turns['gender'].value_counts())
#race distribution
print(df_turns['race'].value_counts())
print(df_turns['proficiency'].value_counts())


In [ ]:
#plot whisker plots, while grouping by imbalance (0,0.25,0.5,0.75)
#get categories of imbalance, slip into 4 classes

#plot just sessions 41 an 38, as a figure with 4 subplots
import tikzplotlib

sns.set(style="ticks", font_scale=2)
#import itertools
plt.rcParams["font.family"] = "Times New Roman"


colors0 = ['royalblue']#,'mediumblue','cornflowerblue']
colors1 = ['indianred']#,'firebrick']
colors2 = ['darkgreen']#,'forestgreen','limegreen']
colors3 = ['darkorange']#,'goldenrod','gold']

imbalance_cat = []
for ind, row in df_turns_diff.iterrows():
    imb = row['imbalance']
    if imb < 0.25:
        imbalance_cat.append('0-0.25')
    elif imb < 0.5:
        imbalance_cat.append('0.25-0.5')
    else:
        imbalance_cat.append('>0.5')

#print the number of elements in each category
print('0-0.25: ', imbalance_cat.count('0-0.25'))
print('0.25-0.5: ', imbalance_cat.count('0.25-0.5'))
print('0.5-0.75: ', imbalance_cat.count('>0.5'))


#add imbalance_cat to df_turns_diff
df_turns_diff['imbalance_cat'] = imbalance_cat

df_turns_sorted = df_turns_diff.sort_values(by = ['imbalance_cat'])

#create figure
plt.figure(figsize=(10, 5))

#create map with colors0, colors1, colors2 and use in plot
palette = dict(zip(df_turns_sorted['imbalance_cat'].unique(), colors0 + colors3 + colors2))

sns.boxplot(x = 'imbalance_cat', y = 'slope_diff',palette=palette, data = df_turns_sorted)

#plt.title('Slope difference between participants, grouped by imbalance')
plt.xlabel('Participation Imbalance')
plt.ylabel('IP Difference')
plt.savefig('../slope_diff_imbalance.png', dpi = 300, bbox_inches='tight')


In [ ]:
#plot slope, other_slope, agreeable_general, other_agreeable, vs. balance_participant and vs. counts_agreement_markers, in a scatter plot

#plot slope vs. imbalance
plt.scatter(df_turns['balance_participant'], df_turns['slope'])
plt.xlabel('balance_participant')
plt.ylabel('slope')
plt.title('slope vs. balance_participant')
plt.show()

#plot other_slope vs. balance_participant
plt.scatter(df_turns['balance_participant'], df_turns['other_slope'])
plt.xlabel('balance_participant')
plt.ylabel('other_slope')
plt.title('other_slope vs. balance_participant')
plt.show()

#plot agreeable_general vs. balance_participant
plt.scatter(df_turns['balance_participant'], df_turns['agreeable_general'])
plt.xlabel('balance_participant')
plt.ylabel('agreeable_general')
plt.title('agreeable_general vs. balance_participant')
plt.show()

#plot other_agreeable vs. imbalance
plt.scatter(df_turns['balance_participant'], df_turns['other_agreeable'])
plt.xlabel('balance_participant')
plt.ylabel('other_agreeable')
plt.title('other_agreeable vs. balance_participant')
plt.show()

#plot slope vs. counts_agreement_markers
plt.scatter(df_turns['counts_agreement_markers'], df_turns['slope'])
plt.xlabel('counts_agreement_markers')
plt.ylabel('slope')
plt.title('slope vs. counts_agreement_markers')
plt.show()

#plot other_slope vs. counts_agreement_markers
plt.scatter(df_turns['counts_agreement_markers'], df_turns['other_slope'])
plt.xlabel('counts_agreement_markers')
plt.ylabel('other_slope')
plt.title('other_slope vs. counts_agreement_markers')
plt.show()

#plot agreeable_general vs. counts_agreement_markers
plt.scatter(df_turns['counts_agreement_markers'], df_turns['agreeable_general'])
plt.xlabel('counts_agreement_markers')
plt.ylabel('agreeable_general')
plt.title('agreeable_general vs. counts_agreement_markers')
plt.show()

#plot other_agreeable vs. counts_agreement_markers
plt.scatter(df_turns['counts_agreement_markers'], df_turns['other_agreeable'])
plt.xlabel('counts_agreement_markers')
plt.ylabel('other_agreeable')
plt.title('other_agreeable vs. counts_agreement_markers')
plt.show()

#plot slope_diff vs. imbalance
plt.scatter(df_turns_diff['imbalance'], df_turns_diff['slope_diff'])
plt.xlabel('imbalance')
plt.ylabel('slope_diff')
plt.title('slope_diff vs. imbalance')
plt.show()



In [ ]:
#plot agreeable_general_diff vs. counts_agreement_markers_diff
plt.scatter(df_turns_diff['counts_agreement_markers_diff'], df_turns_diff['agreeable_general_diff'])
plt.xlabel('counts_agreement_markers_diff')
plt.ylabel('agreeable_general_diff')
plt.title('agreeable_general_diff vs. counts_agreement_markers_diff')
plt.show()


In [ ]:
#plot slope vs counts_agreement_markers
plt.scatter(df_turns['counts_agreement_markers'], df_turns['slope'])
plt.xlabel('counts_agreement_markers')
plt.ylabel('slope')
plt.title('slope vs counts_agreement_markers')
plt.show()

In [ ]:
#plot other_slope vs avg_sentiment_positive
plt.scatter(df_turns['avg_sentiment_positive'], df_turns['other_slope'])
plt.xlabel('avg_sentiment_positive')
plt.ylabel('other_slope')
plt.title('other_slope vs avg_sentiment_positive')
plt.show()


In [ ]:
#barplot with agreeableness and other_slope
sns.barplot(x = 'agreeableness', y = 'other_slope', data = df_turns)

In [ ]:
#barplot with agreeableness and other_slope
sns.barplot(x = 'agreeableness', y = 'other_agreeable', data = df_turns)

In [ ]:
#barplot with agreeableness and other_slope
sns.barplot(x = 'neuroticism', y = 'other_agreeable', data = df_turns)

In [ ]:
#plot slope_diff vs imbalance
plt.scatter(df_turns_diff['imbalance'], df_turns_diff['slope_diff'])
plt.xlabel('imbalance')
plt.ylabel('slope_diff')
plt.title('slope_diff vs imbalance')
plt.show()


In [ ]:
#plot slope_diff vs age_diff
plt.scatter(df_turns_diff['age_diff'], df_turns_diff['slope_diff'])
plt.xlabel('age_diff')
plt.ylabel('slope_diff')
plt.title('slope_diff vs age_diff')
plt.show()

In [ ]:
#plot whisker plots, while grouping by imbalance (0,0.25,0.5,0.75)
#get categories of imbalance, slip into 4 classes
imbalance_cat = []
for ind, row in df_turns_diff.iterrows():
    imb = row['imbalance']
    if imb < 0.25:
        imbalance_cat.append('0-0.25')
    elif imb < 0.5:
        imbalance_cat.append('0.25-0.5')
    elif imb < 0.75:
        imbalance_cat.append('0.5-0.75')
    else:
        imbalance_cat.append('>0.75')


#add imbalance_cat to df_turns_diff
df_turns_diff['imbalance_cat'] = imbalance_cat

df_turns_sorted = df_turns_diff.sort_values(by = ['imbalance_cat'])

sns.boxplot(x = 'imbalance_cat', y = 'slope_diff', data = df_turns_sorted)


In [ ]:
#now, do the same for other_slope vs avg_sentiment_positive
#categorize avg_sentiment_positive
avg_sentiment_positive_cat = []
#sort df_turns by avg_sentiment_positive
df_turns = df_turns.sort_values(by = ['avg_sentiment_positive'])

for ind, row in df_turns.iterrows():
    avg_sent = row['avg_sentiment_positive']
    if avg_sent == 0:
        avg_sentiment_positive_cat.append('0')
    elif avg_sent < 0.03:
        avg_sentiment_positive_cat.append(']0,0.03[')
    elif avg_sent < 0.06:
        avg_sentiment_positive_cat.append('[0.03,0.06[')
    else:
        avg_sentiment_positive_cat.append('[0.06, ]')




#add imbalance_cat to df_turns_diff
df_turns['avg_sentiment_positive_cat'] = avg_sentiment_positive_cat

#sort df by this order: [0, ']0,0.03[','[0.03,0.06[', '[0.06, ]']

#df_turns_sorted = df_turns.sort_values(by = ['avg_sentiment_positive_cat'])

sns.boxplot(x = 'avg_sentiment_positive_cat', y = 'slope', data = df_turns)


In [ ]:
#now, do the same for age_diff vs slope_diff
#categorize age_diff
age_diff_cat = []
#sort df_turns by avg_sentiment_positive
df_turns_diff = df_turns_diff.sort_values(by = ['age_diff'])

age_diff_cat = []
for ind, row in df_turns_diff.iterrows():
    age_diff = row['age_diff']
    if age_diff <10:
        age_diff_cat.append('0-10')
    elif age_diff < 20:
        age_diff_cat.append('10-20')
    elif age_diff < 30:
        age_diff_cat.append('20-30')
    else:
        age_diff_cat.append('>30')



#add imbalance_cat to df_turns_diff
df_turns_diff['age_diff_cat'] = age_diff_cat

#sort df by this order: [0, ']0,0.03[','[0.03,0.06[', '[0.06, ]']

#df_turns_sorted = df_turns.sort_values(by = ['avg_sentiment_positive_cat'])

sns.boxplot(x = 'age_diff_cat', y = 'slope_diff', data = df_turns_diff)


# PCA

In [ ]:
#imports for standartizing dataset
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
#import libraries for running pca analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#immpot pca
from sklearn.decomposition import PCA

In [ ]:
#open data
df = pd.read_csv('../data/all_data_05_norm.csv')
#df_norm = pd.read_csv('../../data/pair_features_overlap_high_discomfort_10hz_normalized.csv')
print(df.shape)
#get columns
cols = df.columns
#print(cols)
#get features, anything from column 5 to the end, excluding cols 6 and 7, adn 124 and 125
features = cols[5:]
features = [f for f in features if f not in ['s2','s3','s120','s121']]

#remove sessions with "_1"
sessions = df['session'].values
sessions = [s for s in sessions if '_1' not in s]
df = df[df['session'].isin(sessions)]
print(df.shape)


#features = [f for f in features if 'r' in f]
print(features)
print(len(features))
classes = cols[2]
participant_col = cols[0]

#we can do all, or participant by participant

#reindex
df = df.reset_index(drop=True)



In [ ]:
#now, run PCA analysis on the data, leave only feature that explain 95% of the variance


x = df[features].values
print(x)


#run PCA analysiS
pca = PCA()
principalComponents = pca.fit_transform(x)

print(principalComponents.shape)

#plot the explained variance
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance')
plt.show()

#keep only the features that explain 90% of the variance
pca = PCA(n_components=0.90)
principalComponents = pca.fit_transform(x)
print('number of components:')
print(principalComponents.shape)


#save the principal components to a file
principalDf = pd.DataFrame(data = principalComponents
             , columns = ['principal_component_' + str(i) for i in range(principalComponents.shape[1])])

#add the initial columns to the principal components
principalDf = pd.concat([df.iloc[:,:5], principalDf], axis = 1)
print(principalDf.head())

#save as csv
principalDf.to_csv('../data/all_data_05_pca.csv', index = False)


In [ ]:
#open data
df = pd.read_csv('../data/all_data_05_norm_sign.csv')
#df_norm = pd.read_csv('../../data/pair_features_overlap_high_discomfort_10hz_normalized.csv')
print(df.shape)
#get columns
cols = df.columns
#print(cols)
#get features, anything from column 5 to the end, excluding cols 6 and 7, adn 124 and 125
features = cols[5:]
features = [f for f in features if f not in ['s2','s3','s120','s121']]

#remove sessions with "_1"
sessions = df['session'].values
sessions = [s for s in sessions if '_1' not in s]
df = df[df['session'].isin(sessions)]
print(df.shape)


#features = [f for f in features if 'r' in f]
print(features)
print(len(features))
classes = cols[2]
participant_col = cols[0]

#we can do all, or participant by participant

#reindex
df = df.reset_index(drop=True)



In [ ]:
#now, run PCA analysis on the data, leave only feature that explain 95% of the variance


x = df[features].values
print(x)


#run PCA analysiS
pca = PCA()
principalComponents = pca.fit_transform(x)

print(principalComponents.shape)

#plot the explained variance
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance')
plt.show()

#keep only the features that explain 90% of the variance
pca = PCA(n_components=0.90)
principalComponents = pca.fit_transform(x)
print('number of components:')
print(principalComponents.shape)


#save the principal components to a file
principalDf = pd.DataFrame(data = principalComponents
             , columns = ['principal_component_' + str(i) for i in range(principalComponents.shape[1])])

#add the initial columns to the principal components
principalDf = pd.concat([df.iloc[:,:5], principalDf], axis = 1)
print(principalDf.head())

#save as csv
principalDf.to_csv('../data/all_data_05_pca_sign.csv', index = False)
